In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from torch.nn.modules import distance

# Data
X, y = make_regression(n_samples=5000, n_features=20, random_state=42)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2)

# Architectures and Wrappers

## Network Architecture and Training Functions

In [ ]:
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, Subset
from torch.nn.utils import clip_grad_norm_
import torch

class MLPRegressor(nn.Module):
    def __init__(self, input_size:int, hidden_size:int, n_hidden:int,  output_size:int):
        super(MLPRegressor, self).__init__()

        self.input_layer = nn.Linear(input_size, hidden_size)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_size, hidden_size) for _ in range(n_hidden)])
        self.output_layer = nn.Linear(hidden_size, output_size)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.input_layer(x))
        for layer in self.hidden_layers:
            x = self.relu(layer(x))

        x = self.output_layer(x)

        return x


## Regressor Ensemble Wrapper

In [ ]:
import copy
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.utils.validation import check_is_fitted
import torch
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Subset, TensorDataset


class RegressionEnsembleWrapper(RegressorMixin, BaseEstimator):

    def __init__(
        self,
        models: list = None,
        batch_size: int = 256,
        lr: float = 3e-4,
        epochs: int = 30,
        device: str = None,
    ):
        """Initializes the regression ensemble wrapper with base neural network models."""
        if device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(device)

        self.models = models if models is not None else []
        self.batch_size = batch_size
        self.epochs = epochs
        self.lr = lr

    def fit(self, X: np.ndarray, y: np.ndarray):
        """Fit all the base estimators on the training data using K-Fold CV."""

        def _get_train_loaders(
            dataset: TensorDataset, train_idx, val_idx, batch_size: int = 256
        ):
            train_loader = DataLoader(
                Subset(dataset, train_idx), batch_size=batch_size, shuffle=True
            )
            val_loader = DataLoader(
                Subset(dataset, val_idx), batch_size=batch_size, shuffle=False
            )
            return train_loader, val_loader

        def _train_one_epoch(
            model, loader, optimizer, criterion, device, max_norm: float = 1.0
        ):
            model.train()
            running_loss = 0.0
            for batch_X, batch_y in loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(batch_X), batch_y)
                running_loss += loss.item()
                loss.backward()
                clip_grad_norm_(model.parameters(), max_norm=max_norm)
                optimizer.step()
            return running_loss / len(loader)

        def _calculate_val_loss(model, loader, criterion, device):
            model.eval()
            running_loss = 0.0
            with torch.no_grad():
                for batch_X, batch_y in loader:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    output = model(batch_X)
                    loss = criterion(output, batch_y)
                    running_loss += loss.item()
            return running_loss / len(loader)

        # FIX: Explicitly parse targets to match MSELoss expected shapes
        y_tensor = (
            torch.Tensor(y).float().unsqueeze(1)
            if y.ndim == 1
            else torch.Tensor(y).float()
        )
        trainval_set = TensorDataset(torch.Tensor(X).float(), y_tensor)

        kf = KFold(n_splits=len(self.models), shuffle=True, random_state=42)
        fitted_models = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
            print(
                f"\n ===================== Training Fold {fold}  ===================== \n"
            )

            train_loader, val_loader = _get_train_loaders(
                dataset=trainval_set,
                train_idx=train_idx,
                val_idx=val_idx,
                batch_size=self.batch_size,
            )

            # Move individual fold model to target device
            model = copy.deepcopy(self.models[fold]).to(self.device)
            optimizer = torch.optim.Adam(model.parameters(), lr=self.lr)
            criterion = torch.nn.MSELoss()

            # FIX: Initialize fold-specific metrics & weights tracking
            best_val_loss = float("inf")
            best_model_wts = copy.deepcopy(model.state_dict())

            for epoch in range(self.epochs):
                train_loss = _train_one_epoch(
                    model, train_loader, optimizer, criterion, self.device
                )
                val_loss = _calculate_val_loss(
                    model, val_loader, criterion, self.device
                )

                # FIX: Capture the exact state dict weights configuration of the best epoch
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_model_wts = copy.deepcopy(model.state_dict())

                if (epoch + 1) % 10 == 0 or epoch == 0:
                    print(
                        f"Epoch {epoch+1:02d}/{self.epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}"
                    )

            # Load the optimal epoch weights back to the model before saving
            model.load_state_dict(best_model_wts)
            fitted_models.append(model)

        self.models = fitted_models
        self.is_fitted_ = True  # Scikit-learn flag convention (ends with trailing underscore)

        return self

    def _mean_response(self, X: np.ndarray) -> np.ndarray:
        """Internal helper for soft voting / averaging regression values across estimators."""
        # Convert NumPy inputs safely to PyTorch evaluation syntax
        X_tensor = torch.Tensor(X).float().to(self.device)
        pred = np.zeros((len(self.models), len(X)))

        for idx, model in enumerate(self.models):
            model.eval()
            with torch.no_grad():
                # Extract output out of tensor context back to clean numpy
                predictions = model(X_tensor).cpu().numpy().flatten()
                pred[idx] = predictions

        return pred.mean(axis=0)

    def predict(self, X: np.ndarray) -> np.ndarray:
        check_is_fitted(self, attributes=["is_fitted_"])
        return self._mean_response(X)

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Returns the R^2 determination score on the given test data and labels."""
        check_is_fitted(self, attributes=["is_fitted_"])
        pred = self.predict(X)
        return r2_score(y, pred)  # Correct sequence order: (y_true, y_pred)

# Model Training

In [ ]:
from sklearn.preprocessing import StandardScaler

model_parameters = {
    'input_size':  X_train_val.shape[1],
    'hidden_size': 64,    # neurons per layer
    'n_hidden':    2,     # number of layers
    'output_size': 1
}

models = []
for i in range(5):
    m = MLPRegressor(**model_parameters)
    models.append(m)



# Scale features
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train_val)
X_test_scaled  = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train_val.reshape(-1, 1)).ravel()
y_test_scaled  = scaler_y.transform(y_test.reshape(-1, 1)).ravel()


regressor = RegressionEnsembleWrapper(
    models=models,
    lr=1e-3,
    epochs=100,
    batch_size=64
)
regressor.fit(X_train_scaled, y_train_scaled)

# Feature Importance and Patrial Dependency Plots

## Permutaion

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

for name, X_data, y_data in [('Treningowy', X_train_val, y_train_val), ('Testowy', X_test_scaled, y_test_scaled)]:
    result = permutation_importance(regressor, X_data, y_data, n_repeats=100, random_state=0, scoring='r2')

    print(f"\n--- Zbiór {name} ---")
    significant_features = []

    for i in range(X_data.shape[1]):
        mean = result.importances_mean[i]
        std = result.importances_std[i]

        # Wyznaczenie 95% przedziału ufności dla średniej
        margin_of_error = 1.96 * (std / np.sqrt(100))
        ci_lower = mean - margin_of_error
        ci_upper = mean + margin_of_error

        if ci_upper < 0 or ci_lower > 0:
            significant_features.append((i, mean, ci_lower, ci_upper))
            print(f"Cecha {i}: Mean={mean:.4f}, CI=[{ci_lower:.4f}, {ci_upper:.4f}]")



feature_names = (
    X_test.columns
    if hasattr(X_test, "columns")
    else [f"Cecha {i}" for i in range(X_test.shape[1])]
)


sorted_importances_idx = result.importances_mean.argsort()[::-1]
sorted_feature_names = [feature_names[i] for i in sorted_importances_idx]
sorted_importances = result.importances[sorted_importances_idx]

plt.figure(figsize=(10, 6))

plt.boxplot(
    sorted_importances.T,
    vert=False,
    tick_labels=sorted_feature_names,
    patch_artist=True,
    boxprops=dict(facecolor="lightblue", color="blue"),
    medianprops=dict(color="red", linewidth=2),
)

plt.axvline(x=0, color="crimson", linestyle="--", linewidth=1.5, label="Próg 0")

plt.title("Rozkład istotności cech (Permutation Importance)", fontsize=14, pad=15)
plt.xlabel("Spadek metryki modelu po przetasowaniu cechy", fontsize=12)
plt.ylabel("Cechy", fontsize=12)
plt.legend(loc="lower right")
plt.tight_layout()

plt.show()

## PDP
Patrial Dependency Plots

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

features = [1,16,17]
PartialDependenceDisplay.from_estimator(regressor, X_test_scaled, features)

features = [(1,16)]
PartialDependenceDisplay.from_estimator(regressor, X_test_scaled, features)

PartialDependenceDisplay.from_estimator(regressor, X_test_scaled, [1], kind='both')

# LIME

## X to Explain
data to model explanation

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

pert_factor = 0.3
num_of_perturbations = 1000
stds = X_train_scaled.std(axis=0)

X_to_explain = X_test[0]
noise = rng.random((X_to_explain.shape[0], num_of_perturbations))*np.expand_dims(std, axis=0)

perturbations = np.expand_dims(X_to_explain, axis=1) + noise
perturbations = perturbations.swapaxes(0, 1)
outputs = regressor.predict(perturbations)

distances = np.exp(-np.linalg.norm(perturbations - X_to_explain,axis=1))

perturbations

In [ ]:
import statsmodels.api as sm

y_lime = outputs.flatten()
X_lime = sm.add_constant(perturbations)

# Weighted Least Squares Model
wls_model = sm.WLS(y_lime, X_lime, weights=distances)
results = wls_model.fit()

print(results.summary())

# Mean Weights for full test dataset

In [ ]:

weights = np.zeros((len(y_lime), X_test_scaled.shape[1]))
for idx in range(len(y_test_scaled)):
    pert_factor = 0.3
    num_of_perturbations = 1000
    stds = X_train_scaled.std(axis=0)

    X_to_explain = X_test[idx]
    noise = rng.random((X_to_explain.shape[0], num_of_perturbations))*np.expand_dims(std, axis=0)

    perturbations = np.expand_dims(X_to_explain, axis=1) + noise
    perturbations = perturbations.swapaxes(0, 1)
    outputs = regressor.predict(perturbations)

    distances = np.exp(-np.linalg.norm(perturbations - X_to_explain,axis=1))

    y_lime = outputs.flatten()
    X_lime = sm.add_constant(perturbations)

    # Weighted Model
    wls_model = sm.WLS(y_lime, X_lime, weights=distances)
    results = wls_model.fit()
    weights[idx] = results.params[1:]

mean_w = weights.mean(axis=0)

print(mean_w)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.barplot(mean_w)
plt.xlabel("Features")
plt.ylabel("Mean weight")
plt.title('LIME model mean weights for test data')
plt.show()

# Shapley

## Shapley I

### X to Explain

In [ ]:
X_to_explain = X_test_scaled[0]
means = X_train_scaled.mean(axis=0)

S = np.arange(20)
N = len(S)
K = 5

m_values = np.random.binomial(N, 0.5, size=(K,)).astype(np.int32)

subsets = []
for m in m_values:
    idx = np.random.permutation(N)[:m]
    subsets.append(S[idx])

masks = np.ones((K, N), dtype=int)

for i, subset in enumerate(subsets):
    if len(subset) > 0:
        masks[i, subset] = 0

print(masks)

In [ ]:
shapley_values = []
for feature_id in range(X_to_explain.shape[0]):
    mask_with_features = masks.copy()
    mask_with_features[:, feature_id] = 1
    permutations = X_to_explain * mask_with_features + means * (1 - mask_with_features)
    outputs_with_feature = regressor.predict(permutations)

    mask_without_features = masks.copy()
    mask_without_features[:, feature_id] = 0
    permutations = X_to_explain * mask_without_features + means * (1 - mask_without_features)
    outputs_without_features = regressor.predict(permutations)

    shap_value = np.mean(outputs_with_feature - outputs_without_features)
    shapley_values.append(shap_value)

sns.barplot(shapley_values)
plt.xlabel('Features')
plt.ylabel('Shap value')
plt.show()

### Mean on test Set

In [ ]:
def shap_values(X: np.ndarray):
    means = X_train_scaled.mean(axis=0)

    S = np.arange(X.shape[0])
    N = len(S)
    K = 5

    m_values = np.random.binomial(N, 0.5, size=(K,)).astype(np.int32)

    subsets = []
    for m in m_values:
        idx = np.random.permutation(N)[:m]
        subsets.append(S[idx])

    masks = np.ones((K, N), dtype=int)

    for i, subset in enumerate(subsets):
        if len(subset) > 0:
            masks[i, subset] = 0

    shapley_values = np.zeros(N)
    for feature_id in range(N):
        mask_with_features = masks.copy()
        mask_with_features[:, feature_id] = 1
        permutations = X * mask_with_features + means * (1 - mask_with_features)
        outputs_with_feature = regressor.predict(permutations)

        mask_without_features = masks.copy()
        mask_without_features[:, feature_id] = 0
        permutations = X * mask_without_features + means * (1 - mask_without_features)
        outputs_without_features = regressor.predict(permutations)

        shap_value = np.mean(outputs_with_feature - outputs_without_features)
        shapley_values[feature_id] = shap_value

    return shapley_values


values = np.zeros(X_test_scaled.shape)
for idx, X in enumerate(X_test_scaled):
    values[idx] = shap_values(X)

print("Shapley Values:"
    f"Mean: {values.mean(axis=0)}\n"
      f"Std: {values.std(axis=0)}\n")


In [ ]:
means = values.mean(axis=0)
sorted_idx = np.argsort(means)
sorted_labels = [str(idx) for idx in sorted_idx]
features = np.arange(len(means))
means = np.sort(means)

sns.barplot(y = means, x = sorted_labels)
# sns.barplot(means)
# plt.xticks(sorted_idx)
plt.xlabel('Features')
plt.ylabel('Shap value')
plt.title('Mean Shap Values on test data')
plt.show()

## Shaply II
Definition Based

In [ ]:
import numpy as np
import math
from itertools import combinations
from sklearn.base import is_classifier, is_regressor

def shapley_value(model, X, feature_idx, predictions_baseline=None):
    """
    Oblicza wartość Shapleya dla danej cechy zgodnie z definicją.

    Parameters:
    -----------
    model : sklearn model
        Wytrenowany model z biblioteki scikit-learn
    X : array-like, shape (n_samples, n_features)
        Dane wejściowe
    feature_idx : int
        Indeks cechy, dla której obliczamy wartość Shapleya
    predictions_baseline : float, optional
        Wartość bazowa (np. średnia predykcja na zbiorze treningowym)
        Jeśli None, zostanie obliczona jako średnia predykcja

    Returns:
    --------
    float : Wartość Shapleya dla danej cechy
    """

    n_features = X.shape[1]
    n_samples = X.shape[0]

    # Uśrednimy obliczenia na wszystkich próbkach
    shapley_values = np.zeros(n_samples)

    for sample_idx in range(n_samples):
        x_sample = X[sample_idx:sample_idx+1].copy()

        # Zbiór wszystkich cech poza feature_idx
        other_features = [i for i in range(n_features) if i != feature_idx]

        total_shapley = 0.0

        # Iterujemy po wszystkich podzbiorach S ⊆ N/i
        for r in range(len(other_features) + 1):
            for S_indices in combinations(other_features, r):
                S = set(S_indices)

                # Obliczenie wagi: |S|! * (|N|-|S|-1)! / |N|!
                s_size = len(S)
                n = n_features
                weight = (math.factorial(s_size) *
                         math.factorial(n - s_size - 1)) / math.factorial(n)

                # Predykcja dla zbioru S (bez feature_idx)
                x_S = x_sample.copy()
                # Ustawiamy cechy spoza S na wartość średnią z zbioru treningowego
                for feat in other_features:
                    if feat not in S:
                        x_S[0, feat] = X[:, feat].mean()

                pred_S = model.predict(x_S)[0]

                # Predykcja dla zbioru S ∪ {feature_idx}
                x_S_union_i = x_sample.copy()
                for feat in other_features:
                    if feat not in S:
                        x_S_union_i[0, feat] = X[:, feat].mean()

                pred_S_union_i = model.predict(x_S_union_i)[0]

                # v(S ∪ {i}) - v(S)
                marginal_contribution = pred_S_union_i - pred_S

                total_shapley += weight * marginal_contribution

        shapley_values[sample_idx] = total_shapley

    # Zwracamy średnią wartość Shapleya na wszystkich próbkach
    return shapley_values.mean()


def shapley_values_all_features(model, X, predictions_baseline=None):
    """
    Oblicza wartości Shapleya dla wszystkich cech.

    Parameters:
    -----------
    model : sklearn model
        Wytrenowany model z biblioteki scikit-learn
    X : array-like, shape (n_samples, n_features)
        Dane wejściowe
    predictions_baseline : float, optional
        Wartość bazowa

    Returns:
    --------
    array, shape (n_features,) : Wartości Shapleya dla każdej cechy
    """

    n_features = X.shape[1]
    shapley_vals = np.zeros(n_features)

    for feature_idx in range(n_features):
        print(f"Obliczanie wartości Shapleya dla cechy {feature_idx}...")
        shapley_vals[feature_idx] = shapley_value(
            model, X, feature_idx, predictions_baseline
        )

    return shapley_vals

In [ ]:
shap_values_def = shapley_values_all_features(regressor, X_test[:100])

In [ ]:
sorted_idx = np.argsort(shap_values_def)
sorted_labels = [str(idx) for idx in sorted_idx]
shap_values_def = np.sort(shap_values_def)

sns.barplot(y = shap_values_def, x = sorted_labels)
# sns.barplot(means)
# plt.xticks(sorted_idx)
plt.xlabel('Features')
plt.ylabel('Shap value')
plt.title('Mean Shap Values on test data')
plt.show()